In [ ]:
import os
import time
import torch
import skimage
import sklearn.metrics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [ ]:
import mnds
import extractor
import detection
import vision_transformer as vit

In [ ]:
PATCH_SIZE = 256
STRIDE = 8
FEATURE_SIZE = 384
TOKENS_PER_PATCH = PATCH_SIZE // STRIDE
DIRECTORY = "/home/caicedo/scr/jcaicedo/Micronuclei-data/"

BATCH_SIZE = 48
EPOCHS = 50
LR = 0.01

device = 'cuda:2' if torch.cuda.is_available() else 'cpu'

# Data loading and preparation

In [ ]:
filelist = os.listdir(DIRECTORY)
annot_files = [x for x in filelist if x.endswith('png')]

training_files = annot_files[0:-1]
validation_files = [annot_files[-1]]

In [ ]:
training_set = mnds.MicronucleiDataset(filelist=training_files, directory=DIRECTORY, mode="random", transform=mnds.detection_transforms)

In [ ]:
validation_set = mnds.MicronucleiDataset(filelist=validation_files, directory=DIRECTORY, mode="fixed")

In [ ]:
len(training_set)

In [ ]:
training_set.all_locs.groupby("Image").count()

In [ ]:
data = np.array([k["coord"] for k in training_set.index])
plt.scatter(data[:,0], data[:,1])

In [ ]:
im, lbl = training_set[4]
print(training_set.W, training_set.H, im.shape, lbl.shape)

fig, ax = plt.subplots(1,2)
ax[0].imshow(lbl)
ax[0].axis('off')
ax[1].imshow(im[0,...])
ax[1].axis('off')

In [ ]:
train_dataloader = DataLoader(training_set, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(validation_set, batch_size=4, shuffle=False)

# Model and training loop

In [ ]:
model = detection.DetectionModel(device=device)
model

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9)

In [ ]:
def train_one_epoch(epoch_index, tb_writer):
    running_loss = 0.
    last_loss = 0.
    
    train_dataloader.dataset.randomize_patch_index()
    for i, data in enumerate(train_dataloader):
        x, y = data
        optimizer.zero_grad()
        p = model(x.to(device))
        
        # Loss function
        Y = torch.reshape(y, (-1, 32*32)).to(device)
        loss = loss_fn(p, Y)
        
        # Training instructions
        loss.backward()
        optimizer.step()
        
        # Report results
        running_loss += loss.item()
#         if i % 10 == 9:
#             last_loss = running_loss / 10
#             print(f' batch {i} loss: {last_loss}')
#             running_loss = 0
    return running_loss / i

In [ ]:
best_vloss = 1_000_000.
epoch_number = 0

start = time.time()
for epoch in range(EPOCHS):
    # Training
    print(f'EPOCH {epoch} - ', end='')
    T = time.time()
    model.train(True)
    avg_loss = train_one_epoch(epoch_number, None)
    
    # Validation
    running_vloss = 0.0
    model.eval()
    with torch.no_grad():
        for i, vdata in enumerate(val_dataloader):
            vin, vls = vdata
            vout = model(vin.to(device))
            Y = torch.reshape(vls, (-1, 32*32)).to(device)
            vloss = loss_fn(vout, Y)
            running_vloss += vloss
    avg_vloss = running_vloss / (i+1)
    C = time.time() - T
    print(f'LOSS: Training: {avg_loss} - Validation: {avg_vloss} - Time: {C:.2f} secs')
    
    epoch_number += 1

C = time.time() - start
print(f"\nTrainined finished in {C:.2f} seconds")

# Validation metrics

In [ ]:
def display_examples(vin, vls, pred):
    for j in range(pred.shape[0]):
        # Visualize predictions
        fig, ax = plt.subplots(1,4)

        ax[0].imshow(pred[j] )
        ax[0].axis('off')

        ax[1].imshow(pred[j] > 0.2)
        ax[1].axis('off')

        ax[2].imshow(vls[j])
        ax[2].axis('off')

        ax[3].imshow(vin[j][0,...])
        ax[3].axis('off')
        plt.show()

In [ ]:
model.eval()

GT = []
PRED = []

with torch.no_grad():
    for i, vdata in enumerate(val_dataloader):
        # Get predictions
        vin, vls = vdata
        pred0 = model(vin.to(device))
        P = torch.reshape(pred0, (-1, 32, 32))
        pred = P.cpu().numpy()
        
        # Collect predictions and ground truth
        PRED.append(pred)
        GT.append(vls.cpu().numpy())
        
        if i % 20 == 0: 
            display_examples(vin, vls, pred)
            
PRED = np.concatenate(PRED, axis=0).reshape((-1,))
GT = np.concatenate(GT, axis=0).reshape((-1,))

In [ ]:
# Precision-recall curve
display = sklearn.metrics.PrecisionRecallDisplay.from_predictions(
    GT, PRED, name="Detector", plot_chance_level=True
)
_ = display.ax_.set_title("Precision-Recall curve")

In [ ]:
# Classification report
report = sklearn.metrics.classification_report(GT, PRED > 0.5)
print(report)